<a href="https://colab.research.google.com/github/xyosho/Agente-IA-de-reparacion-de-celulares-/blob/main/Agente%20reparador%20de%20moviles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mobile Repair Assistant — Cellphone Repair RAG Agent

A self-contained AI agent that assists repair technicians and support staff in diagnosing common cellphone problems (charging issues, black screens, warranty assessment) through a professional Gradio interface.

Built with **Google ADK** (Agent Development Kit), **Gemini**, and a real **RAG** pipeline: the repair knowledge base is written to markdown files, embedded with Gemini embeddings, indexed in **ChromaDB**, and retrieved on demand by the agent through tools.

**Before running (English):** add your Gemini API key to Colab secrets (key panel on the left) with the name `GOOGLE_API_KEY`. Get a free key at https://aistudio.google.com/apikey. Then run all cells (`Runtime → Run all`); the last cell prints a public Gradio link.

**Antes de ejecutar (Español):** agrega tu API key de Gemini a los secretos de Colab (panel de llaves a la izquierda) con el nombre `GOOGLE_API_KEY`. Consigue una gratis en https://aistudio.google.com/apikey. Luego ejecuta todas las celdas (`Entorno de ejecución → Ejecutar todo`); la última celda imprime un enlace público de Gradio.

## 1. Setup — dependencies and API key

In [1]:
%pip install -q "google-adk>=1.4.2" "google-genai>=1.16.0" "chromadb>=0.5.0" "gradio>=6.0.0"
print("Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.6 MB/s eta 0:00:00
Dependencies installed.


In [2]:
import os

# Load the Gemini API key from Colab secrets (or from the environment when running elsewhere)
try:
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    pass  # not running on Colab: expect GOOGLE_API_KEY to be already set

assert os.environ.get("GOOGLE_API_KEY"), (
    "GOOGLE_API_KEY is not set. Add it to Colab secrets (key panel) or export it as an "
    "environment variable. Get a free key at https://aistudio.google.com/apikey"
)
print("GOOGLE_API_KEY configured.")

GOOGLE_API_KEY configured.


## 2. Knowledge base

The repair knowledge base is written to `knowledge_base/*.md`. Each `##` section becomes a retrieval chunk. To extend the assistant, add a new entry to the `KNOWLEDGE_BASE` dictionary (or a new `##` section) and re-run from this cell.

In [3]:
from pathlib import Path

KB_DIR = Path("knowledge_base")
KB_DIR.mkdir(exist_ok=True)

KNOWLEDGE_BASE = {
    "charging_issues.md": """# Charging Issues (Phone Not Charging / Slow Charging)

## Diagnostic Tree: Charging Issue

Follow these steps in order to diagnose a phone that is not charging or charges slowly:

1. Check the charging cable for any visible damage or fraying.
2. Try a different wall adapter and a different power outlet to rule out external power supply problems.
3. Gently clean the charging port on the phone. Use a non-conductive tool like a wooden toothpick or a soft brush, and avoid metal objects that could damage the pins.
4. Restart the phone. Sometimes software glitches can interfere with charging.

## Solutions: Damaged Cable or Adapter

If the cable or wall adapter shows visible damage, replace the faulty charging cable or wall adapter with a certified one that matches the phone's specifications. Uncertified accessories can charge slowly or not at all, and may damage the battery over time.

## Solutions: Dirty Charging Port

If the charging port was dirty, after cleaning the port try charging again. Ensure no lint or debris remains inside the connector. Pocket lint is one of the most common causes of intermittent charging.

## Solutions: Software Glitch Affecting Charging

A simple restart often resolves temporary software issues affecting charging. After restarting, connect the charger again and leave it for a few minutes before checking the battery indicator.

## Solutions: Charging Issue Persists

If the phone still does not charge after checking the cable, adapter, port, and restarting, the issue is likely hardware-related — for example a damaged charging port or an internal battery fault. Professional repair is recommended. If the phone is still under warranty, contact the manufacturer's support before paying for a third-party repair.
""",
    "black_screen.md": """# Black Screen (Phone Is On, But No Display)

## Diagnostic Tree: Black Screen

Follow these steps in order when the phone appears to be on (sounds, vibration) but the screen stays black, or the phone will not turn on:

1. Ensure the phone is charged. Connect it to a known working charger for at least 15-30 minutes to rule out a completely drained battery.
2. Perform a force restart (see the brand-specific button combinations below).
3. Try calling the phone from another device. If it rings or vibrates, the phone's internal functions are working, which points to a display-specific issue.

## Force Restart Button Combinations by Brand

- **iPhone (8 and later):** Quickly press and release Volume Up, then quickly press and release Volume Down, then press and hold the Side button until the Apple logo appears.
- **iPhone 7 / 7 Plus:** Press and hold the Volume Down button and the Side (power) button together until the Apple logo appears.
- **Samsung Galaxy:** Press and hold the Volume Down button and the Power button simultaneously for 10-15 seconds until the device vibrates or the logo appears.
- **Google Pixel:** Press and hold the Power button for about 30 seconds, or Power + Volume Down for 10-15 seconds.
- **Other Android phones (OnePlus, Xiaomi, Motorola, etc.):** Press and hold the Power button and the Volume Down button simultaneously for 10-15 seconds until the device restarts.

## Solutions: Drained Battery

If the battery was completely drained, charge the phone for at least 30 minutes before attempting to power it on again. A deeply discharged battery may take several minutes before the charging indicator appears on screen.

## Solutions: Software Crash / Frozen Screen

Performing a force restart (using the brand-specific combination above) often resolves a frozen or unresponsive screen caused by a software glitch. No data is lost during a force restart.

## Solutions: Black Screen Persists

If the screen remains black even after charging and a force restart, it is likely a hardware malfunction with the display (damaged screen, loose display connector, or a logic-board fault). Professional repair is recommended. Check the warranty status first — display defects not caused by physical damage are usually covered.

## Hard Reset (Factory Reset) — General Guidance

A hard reset (factory reset) erases all data and returns the phone to factory settings. Always back up data first.

- **iPhone:** Settings -> General -> Transfer or Reset iPhone -> Erase All Content and Settings. If the phone is unresponsive, use Recovery Mode via a computer with iTunes/Finder.
- **Android (most brands):** Settings -> System -> Reset options -> Erase all data (factory reset). If the phone will not boot, power off, then hold Power + Volume Up (or Volume Down on some brands) to enter Recovery Mode and choose \"Wipe data / factory reset\".
""",
    "warranty_info.md": """# Warranty Information

## Standard Warranty Coverage

A manufacturer's warranty typically covers defects in materials and workmanship for a period of 12 to 24 months from the original date of purchase, depending on the brand and the country of purchase. Common coverage periods: Apple and Samsung usually offer 12 months (24 months in the European Union); many Android manufacturers offer 12-24 months.

## What Voids the Warranty

The following usually void a manufacturer's warranty:

- Physical damage: drops, cracked screens caused by impact, bent frames.
- Liquid exposure or water damage (unless the failure is unrelated and the device meets its IP rating conditions).
- Unauthorized repairs or opening the device outside official service centers.
- Software modifications such as rooting, jailbreaking, or flashing unofficial firmware (varies by jurisdiction).

## How to Make a Warranty Claim

1. Locate the proof of purchase (receipt or invoice) — it is generally required for any warranty claim.
2. Check the purchase date to confirm the device is within the coverage period.
3. Contact the manufacturer's official support channel or the store where the device was purchased.
4. Describe the defect and the troubleshooting steps already attempted; this speeds up the diagnosis.
5. If accepted, the manufacturer will repair the device, replace it, or refund it according to local consumer law.

## Out-of-Warranty Options

If the device is out of warranty:

- Official paid repair through the manufacturer keeps original parts and certified technicians, but is usually the most expensive option.
- Reputable third-party repair shops are cheaper; ask for a written quote and parts warranty (commonly 90 days on the repair).
- For older devices, compare the repair cost against the device's current value — repairs above 50% of the device value are rarely worth it.
""",
}

for filename, content in KNOWLEDGE_BASE.items():
    (KB_DIR / filename).write_text(content, encoding="utf-8")

print(f"Knowledge base written: {len(KNOWLEDGE_BASE)} documents in {KB_DIR}/")

Knowledge base written: 3 documents in knowledge_base/


## 3. RAG pipeline — chunking, embeddings and vector store

Each markdown document is split into one chunk per `##` section, embedded with Gemini `gemini-embedding-001` and stored in a persistent ChromaDB collection. At query time the question is embedded and the most similar chunks are returned.

In [4]:
import hashlib
import re
from dataclasses import dataclass

import chromadb
from google import genai
from google.genai import types as genai_types

EMBEDDING_MODEL = "gemini-embedding-001"
AGENT_MODEL = "gemini-2.5-flash"
CHROMA_DIR = "chroma_db"
CHROMA_COLLECTION = "repair_knowledge_base"
MAX_WARRANTY_MONTHS = 24

genai_client = genai.Client()


def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT"):
    """Embed a batch of texts with the Gemini embedding model."""
    response = genai_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=genai_types.EmbedContentConfig(task_type=task_type),
    )
    return [embedding.values for embedding in response.embeddings]


@dataclass
class Chunk:
    id: str
    text: str
    source: str
    title: str


def chunk_markdown(text, source):
    """Split a markdown document into one chunk per '##' section."""
    doc_title = ""
    title_match = re.search(r"^# (.+)$", text, flags=re.MULTILINE)
    if title_match:
        doc_title = title_match.group(1).strip()

    sections = re.split(r"^## ", text, flags=re.MULTILINE)
    chunks = []
    for section in sections[1:]:
        heading, _, body = section.partition("\n")
        heading = heading.strip()
        body = body.strip()
        if not body:
            continue
        chunk_text = f"{doc_title} — {heading}\n\n{body}" if doc_title else f"{heading}\n\n{body}"
        chunk_id = hashlib.sha1(f"{source}:{heading}".encode("utf-8")).hexdigest()[:16]
        chunks.append(Chunk(id=chunk_id, text=chunk_text, source=source, title=heading))
    return chunks


def get_collection():
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    return client.get_or_create_collection(
        name=CHROMA_COLLECTION, metadata={"hnsw:space": "cosine"}
    )


def ingest(force=False):
    """Index the knowledge base into ChromaDB. Idempotent unless force=True."""
    collection = get_collection()
    if collection.count() > 0 and not force:
        return 0

    chunks = []
    for path in sorted(KB_DIR.glob("*.md")):
        chunks.extend(chunk_markdown(path.read_text(encoding="utf-8"), source=path.name))
    if not chunks:
        raise RuntimeError(f"No markdown files found in {KB_DIR}")

    embeddings = embed_texts([chunk.text for chunk in chunks], task_type="RETRIEVAL_DOCUMENT")
    collection.upsert(
        ids=[chunk.id for chunk in chunks],
        documents=[chunk.text for chunk in chunks],
        embeddings=embeddings,
        metadatas=[{"source": chunk.source, "title": chunk.title} for chunk in chunks],
    )
    return len(chunks)


def retrieve(query, k=3):
    """Return the k knowledge-base chunks most relevant to the query."""
    collection = get_collection()
    if collection.count() == 0:
        ingest()
        collection = get_collection()

    query_embedding = embed_texts([query], task_type="RETRIEVAL_QUERY")[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for text, metadata, distance in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        hits.append(
            {
                "text": text,
                "source": metadata.get("source", ""),
                "title": metadata.get("title", ""),
                "score": 1.0 - distance,
            }
        )
    return hits


indexed = ingest(force=True)
print(f"Knowledge base indexed: {indexed} chunks.")
print("Sample retrieval:", retrieve("phone not charging", k=1)[0]["title"])

Knowledge base indexed: 15 chunks.
Sample retrieval: Diagnostic Tree: Charging Issue


## 4. Agent tools

Plain Python functions with type hints and docstrings — Google ADK converts them into callable tools automatically. The agent decides when to search the knowledge base and when to assess warranty status.

In [5]:
def search_knowledge_base(query: str) -> str:
    """Search the repair knowledge base for troubleshooting information.

    Use this tool whenever the user describes a phone problem (charging
    issues, black screen, won't turn on, hard reset, warranty rules, etc.)
    to retrieve diagnostic steps and solutions before answering.

    Args:
        query: A short English description of the problem or topic to look
            up, e.g. "phone not charging diagnostic steps" or
            "force restart Samsung".

    Returns:
        The most relevant knowledge-base excerpts, separated by dividers.
    """
    hits = retrieve(query, k=3)
    if not hits:
        return "No relevant information found in the knowledge base."
    sections = [f"[Source: {hit['source']} — {hit['title']}]\n{hit['text']}" for hit in hits]
    return "\n\n---\n\n".join(sections)


def check_warranty(age_months: int) -> str:
    """Estimate whether a phone is still under the manufacturer's warranty.

    Args:
        age_months: The phone's age in months since the original purchase.

    Returns:
        A short assessment of the warranty status.
    """
    if age_months is None or age_months < 0:
        return "Invalid phone age: it must be zero or a positive number of months."
    if age_months <= 12:
        return (
            f"Likely under warranty: the phone is {age_months} months old, within the "
            "standard 12-month manufacturer coverage (24 months in some regions such as the EU)."
        )
    if age_months <= MAX_WARRANTY_MONTHS:
        return (
            f"Possibly under warranty: the phone is {age_months} months old. It exceeds the "
            "standard 12-month coverage but may still be covered in regions with 24-month "
            "warranties (e.g. the EU) or under an extended warranty. Proof of purchase is required."
        )
    return (
        f"Likely out of warranty: the phone is {age_months} months old, beyond the maximum "
        f"typical coverage of {MAX_WARRANTY_MONTHS} months."
    )


print("Tools defined.")
print(check_warranty(18))

Tools defined.
Possibly under warranty: the phone is 18 months old. It exceeds the standard 12-month coverage but may still be covered in regions with 24-month warranties (e.g. the EU) or under an extended warranty. Proof of purchase is required.


## 5. Agent definition and session runner

In [6]:
import asyncio
import uuid

import concurrent.futures
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

DEFAULT_LANGUAGE = "es-LA"

_INSTRUCTION_TEMPLATE = """You are the 'Mobile Repair Assistant', a professional diagnostic \nassistant used by repair technicians and support staff for identifying mobile phones, \ntroubleshooting common issues, and assessing warranty status.

Always answer in {target_language}. Use a professional, technical, and concise tone, as if
writing for a repair-shop work order: numbered diagnostic steps, clear findings, and explicit
recommendations. Do not use emojis.

Conversation context: user messages may start with a bracketed device-context line like
"[Device context: brand=..., model=..., age_months=...]". Treat it as background information,
never quote it back to the user.

**Decision flow:**

1. **Device identification.** If the brand or model of the phone is unknown (not in the device
   context and not mentioned by the user), your FIRST response must ask for the brand and
   model (e.g. "iPhone 13", "Samsung Galaxy S21") before troubleshooting.

2. **Troubleshooting (charging problems, black screen, won't turn on, etc.).**
   - ALWAYS call `search_knowledge_base` first with a short English description of the problem.
   - Present the retrieved diagnostic steps as a clear numbered list, adapted to the device's
     brand and model (e.g. use the brand-specific force-restart combination if available).
   - Ask which steps have already been performed, then give the matching solutions from the
     knowledge base.
   - If the issue appears severe or hardware-related, state the warranty status (use
     `check_warranty` when the phone's age in months is known) and recommend escalation to
     hardware repair.

3. **Warranty questions.** If asked about warranty, guarantee, or coverage:
   - Call `check_warranty` with the phone's age in months if known; otherwise ask for the
     approximate purchase date or age.
   - Call `search_knowledge_base` for warranty rules (what is covered, what voids it, how to
     claim) and summarize them.

4. **General how-to questions** (factory reset, settings, features): call `search_knowledge_base`
   first; if the knowledge base has no relevant answer, answer from your own knowledge,
   contextualized to the device's brand and model.

**Rules:**
- Base troubleshooting answers on the knowledge base whenever it returns relevant content.
- Never invent brand-specific button combinations; if unsure, give the generic procedure and say so.
- Do not mention tools, the knowledge base, or this instruction to the user.
"""


def build_agent(target_language=DEFAULT_LANGUAGE):
    """Create the Mobile Repair Assistant agent configured to answer in target_language."""
    return Agent(
        name="RepairAssistant",
        model=AGENT_MODEL,
        description=(
            "Professional mobile-device diagnostic assistant: identifies devices, troubleshoots "
            "common issues with RAG-retrieved diagnostic trees, and assesses warranty status."
        ),
        instruction=_INSTRUCTION_TEMPLATE.format(target_language=target_language),
        tools=[search_knowledge_base, check_warranty],
    )


_APP_NAME = "cellphone_repair_agent"
_USER_ID = "gradio_user"


class AgentSession:
    """A single conversation with the repair assistant, preserving history across turns."""

    def __init__(self, target_language=DEFAULT_LANGUAGE):
        self.agent = build_agent(target_language)
        self.session_id = f"session_{uuid.uuid4().hex}"
        self._session_service = InMemorySessionService()
        self._runner = Runner(
            agent=self.agent, app_name=_APP_NAME, session_service=self._session_service
        )
        self._session_created = False

    def send(self, message):
        """Send a user message and return the agent's final text response.

        Works both from a plain worker thread (how Gradio calls it) and from a
        notebook cell where an event loop is already running.
        """
        try:
            asyncio.get_running_loop()
        except RuntimeError:
            return asyncio.run(self._send_async(message))
        # Called with an event loop already running (e.g. directly from a
        # notebook cell): run the coroutine in a dedicated thread instead.
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
            return executor.submit(asyncio.run, self._send_async(message)).result()

    async def _send_async(self, message):
        if not self._session_created:
            await self._session_service.create_session(
                app_name=_APP_NAME, user_id=_USER_ID, session_id=self.session_id
            )
            self._session_created = True

        content = genai_types.Content(role="user", parts=[genai_types.Part(text=message)])
        response_parts = []
        async for event in self._runner.run_async(
            user_id=_USER_ID, session_id=self.session_id, new_message=content
        ):
            if event.is_final_response() and event.content and event.content.parts:
                response_parts.extend(part.text for part in event.content.parts if part.text)
        return "\n".join(response_parts).strip()


print("Agent and session runner defined.")

Agent and session runner defined.


## 6. User interface

A professional, technician-oriented Gradio interface: device intake panel with a live warranty status indicator on the left, the diagnostic conversation on the right, and quick-access buttons for the most frequent service queries. The last line prints a public link.

In [ ]:
import gradio as gr

LANGUAGES = {"Español": "es-LA"}

QUICK_QUERIES = [
    ("Falla de carga", "El equipo no carga o carga de forma intermitente. Indícame el árbol de diagnóstico."),
    ("Pantalla negra / no enciende", "El equipo no muestra imagen o no enciende. Indícame el árbol de diagnóstico."),
    ("Estado de garantía", "Verifica el estado de garantía del equipo y qué cubre."),
    ("Restablecimiento de fábrica", "Indícame el procedimiento de restablecimiento de fábrica para este equipo."),
]

_CSS = """
#app-header {padding: 14px 4px 10px 4px; border-bottom: 1px solid var(--border-color-primary);}
#app-header h1 {font-size: 1.35rem; font-weight: 600; margin: 0; letter-spacing: -0.01em;}
#app-header p {color: var(--body-text-color-subdued); margin: 2px 0 0 0; font-size: 0.9rem;}
#intake-panel {border: 1px solid var(--border-color-primary); border-radius: 8px; padding: 14px;}
#intake-panel .label-wrap span, #intake-panel h3 {font-size: 0.95rem;}
#warranty-badge {min-height: 30px;}
#warranty-badge .pill {display: inline-block; padding: 3px 12px; border-radius: 4px;
    font-size: 0.8rem; font-weight: 600; letter-spacing: 0.02em; text-transform: uppercase;}
#warranty-badge .ok {background: rgba(22,163,74,.12); color: #15803d; border: 1px solid rgba(22,163,74,.35);}
#warranty-badge .warn {background: rgba(202,138,4,.12); color: #a16207; border: 1px solid rgba(202,138,4,.35);}
#warranty-badge .out {background: rgba(220,38,38,.10); color: #b91c1c; border: 1px solid rgba(220,38,38,.35);}
#warranty-badge .unknown {background: rgba(100,116,139,.10); color: #64748b; border: 1px solid rgba(100,116,139,.3);}
#app-footer {text-align: center; color: var(--body-text-color-subdued); font-size: 0.78rem;
    padding-top: 10px; border-top: 1px solid var(--border-color-primary); margin-top: 8px;}
"""

_THEME = gr.themes.Base(
    primary_hue="blue",
    secondary_hue="slate",
    neutral_hue="slate",
    radius_size="sm",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
)


def _warranty_badge(age_months):
    if age_months is None or age_months == "":
        return '<span class="pill unknown">Garantía: sin datos</span>'
    age = int(age_months)
    if age < 0:
        return '<span class="pill unknown">Edad inválida</span>'
    if age <= 12:
        return f'<span class="pill ok">En garantía · {age} meses</span>'
    if age <= MAX_WARRANTY_MONTHS:
        return f'<span class="pill warn">Garantía según región · {age} meses</span>'
    return f'<span class="pill out">Fuera de garantía · {age} meses</span>'


def _device_context(brand, model, age_months):
    parts = []
    if brand and brand.strip():
        parts.append(f"brand={brand.strip()}")
    if model and model.strip():
        parts.append(f"model={model.strip()}")
    if age_months is not None and age_months != "":
        parts.append(f"age_months={int(age_months)}")
    return f"[Device context: {', '.join(parts)}]\n" if parts else ""


def build_ui():
    with gr.Blocks(title="Mobile Repair Assistant — Diagnóstico asistido") as demo:
        session_state = gr.State(None)

        gr.HTML(
            """
            <div id="app-header">
                <h1>Mobile Repair Assistant</h1>
                <p>Diagnóstico y reparación de dispositivos móviles asistido por IA · RAG sobre base de conocimiento técnica</p>
            </div>
            """
        )

        with gr.Row(equal_height=False):
            with gr.Column(scale=4, elem_id="intake-panel"):
                gr.Markdown("### Datos del equipo")
                brand_box = gr.Textbox(label="Marca", placeholder="Apple, Samsung, Google…")
                model_box = gr.Textbox(label="Modelo", placeholder="iPhone 13 Pro, Galaxy S22…")
                age_box = gr.Number(
                    label="Antigüedad (meses desde compra)", precision=0, minimum=0, value=None
                )
                warranty_html = gr.HTML(_warranty_badge(None), elem_id="warranty-badge")
                language_box = gr.Dropdown(
                    label="Idioma de respuesta", choices=list(LANGUAGES.keys()), value="Español"
                )
                gr.Markdown("### Consultas frecuentes")
                quick_buttons = [gr.Button(label, size="sm") for label, _ in QUICK_QUERIES]
                clear_btn = gr.Button("Nueva sesión de diagnóstico", variant="stop", size="sm")

            with gr.Column(scale=8):
                chatbot = gr.Chatbot(
                    label="Sesión de diagnóstico",
                    height=520,
                    placeholder=(
                        "Registre los datos del equipo en el panel izquierdo y describa "
                        "la falla reportada para iniciar el diagnóstico."
                    ),
                )
                with gr.Row():
                    msg_box = gr.Textbox(
                        placeholder="Describa la falla o consulta técnica…",
                        show_label=False,
                        scale=8,
                        autofocus=True,
                    )
                    send_btn = gr.Button("Enviar", variant="primary", scale=1)

        gr.HTML(
            '<div id="app-footer">Mobile Repair Assistant · Google ADK · Gemini · ChromaDB · '
            'Las recomendaciones son orientativas; verifique los procedimientos según el fabricante.</div>'
        )

        def respond(message, history, session, brand, model, age, language):
            message = (message or "").strip()
            if not message:
                yield history, session, ""
                return
            history = history + [{"role": "user", "content": message}]
            yield history, session, ""

            if session is None:
                session = AgentSession(LANGUAGES.get(language, DEFAULT_LANGUAGE))
            try:
                reply = session.send(_device_context(brand, model, age) + message)
            except Exception as error:
                reply = f"Se produjo un error al consultar al agente: {error}"
            history = history + [{"role": "assistant", "content": reply}]
            yield history, session, ""

        def reset(_language=None):
            return [], None

        chat_inputs = [msg_box, chatbot, session_state, brand_box, model_box, age_box, language_box]
        chat_outputs = [chatbot, session_state, msg_box]
        msg_box.submit(respond, chat_inputs, chat_outputs)
        send_btn.click(respond, chat_inputs, chat_outputs)

        clear_btn.click(reset, None, [chatbot, session_state])
        # Changing the answer language starts a fresh session so the agent instruction matches.
        language_box.change(reset, language_box, [chatbot, session_state])

        age_box.change(_warranty_badge, age_box, warranty_html)

        for button, (_, query_text) in zip(quick_buttons, QUICK_QUERIES):
            button.click(lambda q=query_text: q, None, msg_box)

    return demo


demo = build_ui()
demo.launch(theme=_THEME, css=_CSS, share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e983ea59d9a6dab777.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/di